# Notebook 4: Model Comparison & RBAC

## Part 1: Model Comparison

6 models tested with identical prompts. Only the model parameter changes.

| Model | Tier | Cost |
|-------|------|------|
| openai-gpt-5-mini | Budget | $ |
| llama3.3-70b | Open-source | $ |
| mistral-large2 | Mid-range | $$ |
| claude-sonnet-4-6 | Premium | $$$ |
| openai-gpt-5 | Premium | $$$ |
| claude-opus-4-6 | High-end | $$$$ |

---

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
try:
    state = session.sql("SELECT NOTEBOOK_ID FROM AI_STUDIO_DEMO.PUBLIC.DEMO_STATE WHERE NOTEBOOK_ID = 'evaluation'").collect()
    if not state: raise Exception()
    print('Prerequisite check passed: Notebook 3 (Evaluation) completed.')
except:
    print('WARNING: Please run Notebook 3 (Evaluation & Iteration) first!')

In [ ]:
USE DATABASE AI_STUDIO_DEMO;
USE SCHEMA PUBLIC;

## Create Model-Specific Functions

Each function uses `CALL SNOWFLAKE.CORTEX.CREATE_AI_FUNCTION()` with the identical prompt and outputs. Only the MODEL parameter (param 2) changes.

In [ ]:
-- Create all 6 model-specific functions via CREATE_AI_FUNCTION

-- CLASSIFY_TICKET_GPT5_MINI (Budget: openai-gpt-5-mini)
CALL SNOWFLAKE.CORTEX.CREATE_AI_FUNCTION(
    'AI_STUDIO_DEMO.PUBLIC.CLASSIFY_TICKET_GPT5_MINI',
    'openai-gpt-5-mini',
    $$You are an enterprise support ticket classifier. Classify using EXACT enum values only.

division: "Industrial Adhesives & Tapes", "Safety & Industrial", "Electronics & Energy", "Healthcare", "Transportation & Electronics", "Consumer"
issue_type: "Product Defect / Failure", "Technical Specification Inquiry", "Custom Formulation Request", "Supply Chain / Delivery", "Regulatory / Compliance", "Warranty / RMA", "Pricing / Contract Dispute", "Sample Request", "Application Engineering Support"
priority: "P1-Critical" (line stopped/safety/<72h), "P2-High" (quality/VP/SLA), "P3-Standard" (inquiry/RMA/samples), "P4-Low" (general/catalog)
customer_segment: "OEM", "Distributor", "Converter", "End User", "Government/Defense"
escalation_indicators: "production_line_down", "safety_incident", "regulatory_deadline", "executive_involvement", "contractual_risk"
routing_queue: DIVISION_CODE-FUNCTION-SECTOR (IAT,SI,EE,HC,TE,CON + AppEng,QA,TechServ,Regulatory,Sales,Warranty,Logistics)
suggested_response_template: ACK-P1-DEFECT, ACK-P1-SAFETY, ACK-P2-QUALITY, ACK-P2-REGULATORY, ACK-SPEC-REQUEST, ACK-SAMPLE-REQUEST, ACK-RMA, ACK-WARRANTY, ACK-APPENG, ACK-GENERAL, ACK-PRICING$$,
    $$Classify this ticket:

{TICKET_TEXT}$$,
    PARSE_JSON('[{"name":"TICKET_TEXT","sql_type":"VARCHAR"}]'),
    PARSE_JSON('[{"name":"division","json_type":"string","description":"Exact enum"},{"name":"issue_type","json_type":"string","description":"Exact enum"},{"name":"priority","json_type":"string","description":"P1-P4"},{"name":"priority_reasoning","json_type":"string","description":"Justification"},{"name":"routing_queue","json_type":"string","description":"CODE-FUNCTION-SECTOR"},{"name":"product_family","json_type":"string","description":"Product family"},{"name":"customer_segment","json_type":"string","description":"Exact enum"},{"name":"sla_flag","json_type":"boolean","description":"SLA indicator"},{"name":"escalation_indicators","json_type":"array","description":"From allowed list"},{"name":"suggested_response_template","json_type":"string","description":"ACK template ID"}]'),
    'Model comparison - openai-gpt-5-mini',
    NULL,
    NULL
);

-- CLASSIFY_TICKET_LLAMA33 (Open-source: llama3.3-70b)
CALL SNOWFLAKE.CORTEX.CREATE_AI_FUNCTION(
    'AI_STUDIO_DEMO.PUBLIC.CLASSIFY_TICKET_LLAMA33',
    'llama3.3-70b',
    $$You are an enterprise support ticket classifier. Classify using EXACT enum values only.

division: "Industrial Adhesives & Tapes", "Safety & Industrial", "Electronics & Energy", "Healthcare", "Transportation & Electronics", "Consumer"
issue_type: "Product Defect / Failure", "Technical Specification Inquiry", "Custom Formulation Request", "Supply Chain / Delivery", "Regulatory / Compliance", "Warranty / RMA", "Pricing / Contract Dispute", "Sample Request", "Application Engineering Support"
priority: "P1-Critical" (line stopped/safety/<72h), "P2-High" (quality/VP/SLA), "P3-Standard" (inquiry/RMA/samples), "P4-Low" (general/catalog)
customer_segment: "OEM", "Distributor", "Converter", "End User", "Government/Defense"
escalation_indicators: "production_line_down", "safety_incident", "regulatory_deadline", "executive_involvement", "contractual_risk"
routing_queue: DIVISION_CODE-FUNCTION-SECTOR (IAT,SI,EE,HC,TE,CON + AppEng,QA,TechServ,Regulatory,Sales,Warranty,Logistics)
suggested_response_template: ACK-P1-DEFECT, ACK-P1-SAFETY, ACK-P2-QUALITY, ACK-P2-REGULATORY, ACK-SPEC-REQUEST, ACK-SAMPLE-REQUEST, ACK-RMA, ACK-WARRANTY, ACK-APPENG, ACK-GENERAL, ACK-PRICING$$,
    $$Classify this ticket:

{TICKET_TEXT}$$,
    PARSE_JSON('[{"name":"TICKET_TEXT","sql_type":"VARCHAR"}]'),
    PARSE_JSON('[{"name":"division","json_type":"string","description":"Exact enum"},{"name":"issue_type","json_type":"string","description":"Exact enum"},{"name":"priority","json_type":"string","description":"P1-P4"},{"name":"priority_reasoning","json_type":"string","description":"Justification"},{"name":"routing_queue","json_type":"string","description":"CODE-FUNCTION-SECTOR"},{"name":"product_family","json_type":"string","description":"Product family"},{"name":"customer_segment","json_type":"string","description":"Exact enum"},{"name":"sla_flag","json_type":"boolean","description":"SLA indicator"},{"name":"escalation_indicators","json_type":"array","description":"From allowed list"},{"name":"suggested_response_template","json_type":"string","description":"ACK template ID"}]'),
    'Model comparison - llama3.3-70b',
    NULL,
    NULL
);

-- CLASSIFY_TICKET_MISTRAL (Mid-range: mistral-large2)
CALL SNOWFLAKE.CORTEX.CREATE_AI_FUNCTION(
    'AI_STUDIO_DEMO.PUBLIC.CLASSIFY_TICKET_MISTRAL',
    'mistral-large2',
    $$You are an enterprise support ticket classifier. Classify using EXACT enum values only.

division: "Industrial Adhesives & Tapes", "Safety & Industrial", "Electronics & Energy", "Healthcare", "Transportation & Electronics", "Consumer"
issue_type: "Product Defect / Failure", "Technical Specification Inquiry", "Custom Formulation Request", "Supply Chain / Delivery", "Regulatory / Compliance", "Warranty / RMA", "Pricing / Contract Dispute", "Sample Request", "Application Engineering Support"
priority: "P1-Critical" (line stopped/safety/<72h), "P2-High" (quality/VP/SLA), "P3-Standard" (inquiry/RMA/samples), "P4-Low" (general/catalog)
customer_segment: "OEM", "Distributor", "Converter", "End User", "Government/Defense"
escalation_indicators: "production_line_down", "safety_incident", "regulatory_deadline", "executive_involvement", "contractual_risk"
routing_queue: DIVISION_CODE-FUNCTION-SECTOR (IAT,SI,EE,HC,TE,CON + AppEng,QA,TechServ,Regulatory,Sales,Warranty,Logistics)
suggested_response_template: ACK-P1-DEFECT, ACK-P1-SAFETY, ACK-P2-QUALITY, ACK-P2-REGULATORY, ACK-SPEC-REQUEST, ACK-SAMPLE-REQUEST, ACK-RMA, ACK-WARRANTY, ACK-APPENG, ACK-GENERAL, ACK-PRICING$$,
    $$Classify this ticket:

{TICKET_TEXT}$$,
    PARSE_JSON('[{"name":"TICKET_TEXT","sql_type":"VARCHAR"}]'),
    PARSE_JSON('[{"name":"division","json_type":"string","description":"Exact enum"},{"name":"issue_type","json_type":"string","description":"Exact enum"},{"name":"priority","json_type":"string","description":"P1-P4"},{"name":"priority_reasoning","json_type":"string","description":"Justification"},{"name":"routing_queue","json_type":"string","description":"CODE-FUNCTION-SECTOR"},{"name":"product_family","json_type":"string","description":"Product family"},{"name":"customer_segment","json_type":"string","description":"Exact enum"},{"name":"sla_flag","json_type":"boolean","description":"SLA indicator"},{"name":"escalation_indicators","json_type":"array","description":"From allowed list"},{"name":"suggested_response_template","json_type":"string","description":"ACK template ID"}]'),
    'Model comparison - mistral-large2',
    NULL,
    NULL
);

-- CLASSIFY_TICKET_SONNET (Premium: claude-sonnet-4-6)
CALL SNOWFLAKE.CORTEX.CREATE_AI_FUNCTION(
    'AI_STUDIO_DEMO.PUBLIC.CLASSIFY_TICKET_SONNET',
    'claude-sonnet-4-6',
    $$You are an enterprise support ticket classifier. Classify using EXACT enum values only.

division: "Industrial Adhesives & Tapes", "Safety & Industrial", "Electronics & Energy", "Healthcare", "Transportation & Electronics", "Consumer"
issue_type: "Product Defect / Failure", "Technical Specification Inquiry", "Custom Formulation Request", "Supply Chain / Delivery", "Regulatory / Compliance", "Warranty / RMA", "Pricing / Contract Dispute", "Sample Request", "Application Engineering Support"
priority: "P1-Critical" (line stopped/safety/<72h), "P2-High" (quality/VP/SLA), "P3-Standard" (inquiry/RMA/samples), "P4-Low" (general/catalog)
customer_segment: "OEM", "Distributor", "Converter", "End User", "Government/Defense"
escalation_indicators: "production_line_down", "safety_incident", "regulatory_deadline", "executive_involvement", "contractual_risk"
routing_queue: DIVISION_CODE-FUNCTION-SECTOR (IAT,SI,EE,HC,TE,CON + AppEng,QA,TechServ,Regulatory,Sales,Warranty,Logistics)
suggested_response_template: ACK-P1-DEFECT, ACK-P1-SAFETY, ACK-P2-QUALITY, ACK-P2-REGULATORY, ACK-SPEC-REQUEST, ACK-SAMPLE-REQUEST, ACK-RMA, ACK-WARRANTY, ACK-APPENG, ACK-GENERAL, ACK-PRICING$$,
    $$Classify this ticket:

{TICKET_TEXT}$$,
    PARSE_JSON('[{"name":"TICKET_TEXT","sql_type":"VARCHAR"}]'),
    PARSE_JSON('[{"name":"division","json_type":"string","description":"Exact enum"},{"name":"issue_type","json_type":"string","description":"Exact enum"},{"name":"priority","json_type":"string","description":"P1-P4"},{"name":"priority_reasoning","json_type":"string","description":"Justification"},{"name":"routing_queue","json_type":"string","description":"CODE-FUNCTION-SECTOR"},{"name":"product_family","json_type":"string","description":"Product family"},{"name":"customer_segment","json_type":"string","description":"Exact enum"},{"name":"sla_flag","json_type":"boolean","description":"SLA indicator"},{"name":"escalation_indicators","json_type":"array","description":"From allowed list"},{"name":"suggested_response_template","json_type":"string","description":"ACK template ID"}]'),
    'Model comparison - claude-sonnet-4-6',
    NULL,
    NULL
);

-- CLASSIFY_TICKET_GPT5 (Premium: openai-gpt-5)
CALL SNOWFLAKE.CORTEX.CREATE_AI_FUNCTION(
    'AI_STUDIO_DEMO.PUBLIC.CLASSIFY_TICKET_GPT5',
    'openai-gpt-5',
    $$You are an enterprise support ticket classifier. Classify using EXACT enum values only.

division: "Industrial Adhesives & Tapes", "Safety & Industrial", "Electronics & Energy", "Healthcare", "Transportation & Electronics", "Consumer"
issue_type: "Product Defect / Failure", "Technical Specification Inquiry", "Custom Formulation Request", "Supply Chain / Delivery", "Regulatory / Compliance", "Warranty / RMA", "Pricing / Contract Dispute", "Sample Request", "Application Engineering Support"
priority: "P1-Critical" (line stopped/safety/<72h), "P2-High" (quality/VP/SLA), "P3-Standard" (inquiry/RMA/samples), "P4-Low" (general/catalog)
customer_segment: "OEM", "Distributor", "Converter", "End User", "Government/Defense"
escalation_indicators: "production_line_down", "safety_incident", "regulatory_deadline", "executive_involvement", "contractual_risk"
routing_queue: DIVISION_CODE-FUNCTION-SECTOR (IAT,SI,EE,HC,TE,CON + AppEng,QA,TechServ,Regulatory,Sales,Warranty,Logistics)
suggested_response_template: ACK-P1-DEFECT, ACK-P1-SAFETY, ACK-P2-QUALITY, ACK-P2-REGULATORY, ACK-SPEC-REQUEST, ACK-SAMPLE-REQUEST, ACK-RMA, ACK-WARRANTY, ACK-APPENG, ACK-GENERAL, ACK-PRICING$$,
    $$Classify this ticket:

{TICKET_TEXT}$$,
    PARSE_JSON('[{"name":"TICKET_TEXT","sql_type":"VARCHAR"}]'),
    PARSE_JSON('[{"name":"division","json_type":"string","description":"Exact enum"},{"name":"issue_type","json_type":"string","description":"Exact enum"},{"name":"priority","json_type":"string","description":"P1-P4"},{"name":"priority_reasoning","json_type":"string","description":"Justification"},{"name":"routing_queue","json_type":"string","description":"CODE-FUNCTION-SECTOR"},{"name":"product_family","json_type":"string","description":"Product family"},{"name":"customer_segment","json_type":"string","description":"Exact enum"},{"name":"sla_flag","json_type":"boolean","description":"SLA indicator"},{"name":"escalation_indicators","json_type":"array","description":"From allowed list"},{"name":"suggested_response_template","json_type":"string","description":"ACK template ID"}]'),
    'Model comparison - openai-gpt-5',
    NULL,
    NULL
);

-- CLASSIFY_TICKET_OPUS (High-end: claude-opus-4-6)
CALL SNOWFLAKE.CORTEX.CREATE_AI_FUNCTION(
    'AI_STUDIO_DEMO.PUBLIC.CLASSIFY_TICKET_OPUS',
    'claude-opus-4-6',
    $$You are an enterprise support ticket classifier. Classify using EXACT enum values only.

division: "Industrial Adhesives & Tapes", "Safety & Industrial", "Electronics & Energy", "Healthcare", "Transportation & Electronics", "Consumer"
issue_type: "Product Defect / Failure", "Technical Specification Inquiry", "Custom Formulation Request", "Supply Chain / Delivery", "Regulatory / Compliance", "Warranty / RMA", "Pricing / Contract Dispute", "Sample Request", "Application Engineering Support"
priority: "P1-Critical" (line stopped/safety/<72h), "P2-High" (quality/VP/SLA), "P3-Standard" (inquiry/RMA/samples), "P4-Low" (general/catalog)
customer_segment: "OEM", "Distributor", "Converter", "End User", "Government/Defense"
escalation_indicators: "production_line_down", "safety_incident", "regulatory_deadline", "executive_involvement", "contractual_risk"
routing_queue: DIVISION_CODE-FUNCTION-SECTOR (IAT,SI,EE,HC,TE,CON + AppEng,QA,TechServ,Regulatory,Sales,Warranty,Logistics)
suggested_response_template: ACK-P1-DEFECT, ACK-P1-SAFETY, ACK-P2-QUALITY, ACK-P2-REGULATORY, ACK-SPEC-REQUEST, ACK-SAMPLE-REQUEST, ACK-RMA, ACK-WARRANTY, ACK-APPENG, ACK-GENERAL, ACK-PRICING$$,
    $$Classify this ticket:

{TICKET_TEXT}$$,
    PARSE_JSON('[{"name":"TICKET_TEXT","sql_type":"VARCHAR"}]'),
    PARSE_JSON('[{"name":"division","json_type":"string","description":"Exact enum"},{"name":"issue_type","json_type":"string","description":"Exact enum"},{"name":"priority","json_type":"string","description":"P1-P4"},{"name":"priority_reasoning","json_type":"string","description":"Justification"},{"name":"routing_queue","json_type":"string","description":"CODE-FUNCTION-SECTOR"},{"name":"product_family","json_type":"string","description":"Product family"},{"name":"customer_segment","json_type":"string","description":"Exact enum"},{"name":"sla_flag","json_type":"boolean","description":"SLA indicator"},{"name":"escalation_indicators","json_type":"array","description":"From allowed list"},{"name":"suggested_response_template","json_type":"string","description":"ACK template ID"}]'),
    'Model comparison - claude-opus-4-6',
    NULL,
    NULL
);



## Materialize Results

Run all 6 models on the 12-row test set and store predictions (avoids re-running inference when analyzing).

In [ ]:
CREATE OR REPLACE TABLE MODEL_COMPARISON_RESULTS AS
SELECT t.TEST_ID, t.EXPECTED_OUTPUT, 'openai-gpt-5-mini' AS model_name, CLASSIFY_TICKET_GPT5_MINI(t.TICKET_TEXT) AS predicted FROM EVALUATION_TEST_DATA t;

INSERT INTO MODEL_COMPARISON_RESULTS
SELECT t.TEST_ID, t.EXPECTED_OUTPUT, 'llama3.3-70b', CLASSIFY_TICKET_LLAMA33(t.TICKET_TEXT) FROM EVALUATION_TEST_DATA t;

INSERT INTO MODEL_COMPARISON_RESULTS
SELECT t.TEST_ID, t.EXPECTED_OUTPUT, 'mistral-large2', CLASSIFY_TICKET_MISTRAL(t.TICKET_TEXT) FROM EVALUATION_TEST_DATA t;

INSERT INTO MODEL_COMPARISON_RESULTS
SELECT t.TEST_ID, t.EXPECTED_OUTPUT, 'claude-sonnet-4-6', CLASSIFY_TICKET_SONNET(t.TICKET_TEXT) FROM EVALUATION_TEST_DATA t;

INSERT INTO MODEL_COMPARISON_RESULTS
SELECT t.TEST_ID, t.EXPECTED_OUTPUT, 'openai-gpt-5', CLASSIFY_TICKET_GPT5(t.TICKET_TEXT) FROM EVALUATION_TEST_DATA t;

INSERT INTO MODEL_COMPARISON_RESULTS
SELECT t.TEST_ID, t.EXPECTED_OUTPUT, 'claude-opus-4-6', CLASSIFY_TICKET_OPUS(t.TICKET_TEXT) FROM EVALUATION_TEST_DATA t;

## Model Scorecard

In [ ]:
SELECT
  model_name,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:priority::VARCHAR = predicted:priority::VARCHAR, 1.0, 0.0)), 3) AS priority_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:division::VARCHAR = predicted:division::VARCHAR, 1.0, 0.0)), 3) AS division_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:issue_type::VARCHAR = predicted:issue_type::VARCHAR, 1.0, 0.0)), 3) AS issue_type_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:customer_segment::VARCHAR = predicted:customer_segment::VARCHAR, 1.0, 0.0)), 3) AS segment_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:sla_flag::BOOLEAN = predicted:sla_flag::BOOLEAN, 1.0, 0.0)), 3) AS sla_flag_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:priority::VARCHAR = predicted:priority::VARCHAR, 1.0, 0.0))*0.4 + AVG(IFF(EXPECTED_OUTPUT:division::VARCHAR = predicted:division::VARCHAR, 1.0, 0.0))*0.3 + AVG(IFF(EXPECTED_OUTPUT:issue_type::VARCHAR = predicted:issue_type::VARCHAR, 1.0, 0.0))*0.2 + AVG(IFF(EXPECTED_OUTPUT:sla_flag::BOOLEAN = predicted:sla_flag::BOOLEAN, 1.0, 0.0))*0.1, 3) AS weighted_composite
FROM MODEL_COMPARISON_RESULTS
GROUP BY model_name
ORDER BY weighted_composite DESC;

## P1 Safety Deep-Dive

Which models misclassify P1-Critical tickets?

In [ ]:
SELECT model_name, TEST_ID,
  EXPECTED_OUTPUT:priority::VARCHAR AS expected,
  predicted:priority::VARCHAR AS predicted
FROM MODEL_COMPARISON_RESULTS
WHERE EXPECTED_OUTPUT:priority::VARCHAR = 'P1-Critical'
  AND predicted:priority::VARCHAR != 'P1-Critical'
ORDER BY model_name, TEST_ID;

---
## Part 2: RBAC — Securing AI Functions

### Approach 1: Account-Level Roles (Simple)

In [ ]:
CREATE ROLE IF NOT EXISTS SUPPORT_TICKET_ADMIN COMMENT = 'Manage AI functions';
CREATE ROLE IF NOT EXISTS SUPPORT_TICKET_ANALYST COMMENT = 'Execute AI functions';
GRANT ROLE SUPPORT_TICKET_ANALYST TO ROLE SUPPORT_TICKET_ADMIN;
GRANT ROLE SUPPORT_TICKET_ADMIN TO ROLE SYSADMIN;
GRANT USAGE ON DATABASE AI_STUDIO_DEMO TO ROLE SUPPORT_TICKET_ANALYST;
GRANT USAGE ON SCHEMA AI_STUDIO_DEMO.PUBLIC TO ROLE SUPPORT_TICKET_ANALYST;
GRANT USAGE ON FUNCTION AI_STUDIO_DEMO.PUBLIC.CLASSIFY_SUPPORT_TICKET_V2(VARCHAR) TO ROLE SUPPORT_TICKET_ANALYST;
GRANT SELECT ON TABLE AI_STUDIO_DEMO.PUBLIC.SUPPORT_TICKETS TO ROLE SUPPORT_TICKET_ANALYST;
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE SUPPORT_TICKET_ANALYST;

### Approach 2: Database Roles → Account Roles (Enterprise)

Database roles travel with clones and can be granted to shares.

In [ ]:
CREATE DATABASE ROLE IF NOT EXISTS AI_STUDIO_DEMO.AI_FUNC_EXECUTOR COMMENT = 'Execute AI functions';
GRANT USAGE ON SCHEMA AI_STUDIO_DEMO.PUBLIC TO DATABASE ROLE AI_STUDIO_DEMO.AI_FUNC_EXECUTOR;
GRANT USAGE ON FUNCTION AI_STUDIO_DEMO.PUBLIC.CLASSIFY_SUPPORT_TICKET_V2(VARCHAR) TO DATABASE ROLE AI_STUDIO_DEMO.AI_FUNC_EXECUTOR;
GRANT SELECT ON TABLE AI_STUDIO_DEMO.PUBLIC.SUPPORT_TICKETS TO DATABASE ROLE AI_STUDIO_DEMO.AI_FUNC_EXECUTOR;
GRANT USAGE ON FUTURE FUNCTIONS IN SCHEMA AI_STUDIO_DEMO.PUBLIC TO DATABASE ROLE AI_STUDIO_DEMO.AI_FUNC_EXECUTOR;

-- Bridge to account role
CREATE ROLE IF NOT EXISTS SUPPORT_AI_CONSUMER_ROLE COMMENT = 'Account role for AI function consumers';
GRANT DATABASE ROLE AI_STUDIO_DEMO.AI_FUNC_EXECUTOR TO ROLE SUPPORT_AI_CONSUMER_ROLE;
GRANT USAGE ON DATABASE AI_STUDIO_DEMO TO ROLE SUPPORT_AI_CONSUMER_ROLE;
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE SUPPORT_AI_CONSUMER_ROLE;
GRANT ROLE SUPPORT_AI_CONSUMER_ROLE TO ROLE SYSADMIN;

| Consideration | Account Roles | Database Roles |
|---------------|---------------|----------------|
| Portability (clone/share) | No | Yes |
| Multi-env consistency | Manual | Automatic via clone |
| Best for | Small teams | Enterprise governance |

---
## Demo Complete

In [ ]:
MERGE INTO DEMO_STATE AS t
USING (SELECT 'model_comparison' AS NOTEBOOK_ID, CURRENT_TIMESTAMP() AS COMPLETED_AT) AS s
ON t.NOTEBOOK_ID = s.NOTEBOOK_ID
WHEN MATCHED THEN UPDATE SET COMPLETED_AT = s.COMPLETED_AT
WHEN NOT MATCHED THEN INSERT (NOTEBOOK_ID, COMPLETED_AT) VALUES (s.NOTEBOOK_ID, s.COMPLETED_AT);